# Example: Production RAG with Python
Companion notebook for the **Example: Production RAG with Python** section of Chapter 10 -- the capstone that connects the whole chapter: ingest documents, build dual indexes, run filtered hybrid retrieval, rerank, and call the same pattern through a real MCP server.

This is the book's `chunk_doc`, unchanged:

In [1]:
from typing import List, Dict


def chunk_doc(doc: Dict) -> List[Dict]:
    text = doc["text"]
    size, overlap = 500, 50
    chunks = []
    start = 0
    i = 0
    while start < len(text):
        end = min(len(text), start + size)
        chunks.append({
            "doc_id": doc["doc_id"],
            "chunk_id": f'{doc["doc_id"]}:{i}',
            "text": text[start:end],
            "tenant_id": doc["tenant_id"],
            "source": doc["source"],
            "access": doc["access"],
        })
        start += size - overlap
        i += 1
    return chunks

A deterministic embedder (notebook 02) and two tiny indexes standing in for a real vector store and a real BM25-style keyword store. Both stem a trailing "s" before matching (e.g. "refund" / "refunds") -- the same simplification a real system handles through subword tokenization:

In [2]:
import hashlib
import math

EMBED_DIM = 32


def _stem(word: str) -> str:
    return word[:-1] if len(word) > 3 and word.endswith("s") else word


def embed_fn(text: str) -> list[float]:
    vector = [0.0] * EMBED_DIM
    for word in text.lower().split():
        idx = int(hashlib.sha256(_stem(word).encode()).hexdigest(), 16) % EMBED_DIM
        vector[idx] += 1.0
    norm = math.sqrt(sum(v * v for v in vector)) or 1.0
    return [v / norm for v in vector]


def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b))

In [3]:
class VectorIndex:
    def __init__(self):
        self._store = {}

    def upsert(self, chunk_id, vec, metadata):
        self._store[chunk_id] = {"vec": vec, "metadata": metadata}

    def search(self, query_vec, top_k, filters):
        results = []
        for chunk_id, entry in self._store.items():
            md = entry["metadata"]
            if md["tenant_id"] != filters["tenant_id"] or md["access"] not in filters["access"]["$in"]:
                continue
            results.append({"chunk_id": chunk_id, "score": cosine_similarity(query_vec, entry["vec"])})
        results.sort(key=lambda r: r["score"], reverse=True)
        return results[:top_k]


class KeywordIndex:
    def __init__(self):
        self._store = {}

    def add(self, chunk_id, text, metadata):
        self._store[chunk_id] = {"text": text, "metadata": metadata}

    def search(self, query, top_k, filters):
        q_words = {_stem(w) for w in query.lower().split()}
        results = []
        for chunk_id, entry in self._store.items():
            md = entry["metadata"]
            if md["tenant_id"] != filters["tenant_id"] or md["access"] not in filters["access"]["$in"]:
                continue
            text_words = {_stem(w) for w in entry["text"].lower().split()}
            overlap = len(q_words & text_words)
            score = overlap / max(len(q_words), 1)
            if score > 0:
                results.append({"chunk_id": chunk_id, "score": score})
        results.sort(key=lambda r: r["score"], reverse=True)
        return results[:top_k]

The merged `candidates` list `search()` builds only carries `chunk_id` and `score` -- no chunk text -- so a real reranker needs a way to look the text back up before it can score anything. This one holds a small `chunk_id -> text` lookup for exactly that:

In [4]:
class Reranker:
    def __init__(self, chunk_lookup: dict[str, str]):
        self._chunk_lookup = chunk_lookup

    def rerank(self, query, candidates):
        q_words = {_stem(w) for w in query.lower().split()}
        scored = []
        for c in candidates:
            text = self._chunk_lookup.get(c["chunk_id"], "")
            overlap = len(q_words & {_stem(w) for w in text.lower().split()})
            scored.append({**c, "score": overlap / max(len(q_words), 1)})
        scored.sort(key=lambda r: r["score"], reverse=True)
        return scored

This is the book's `ingest`, unchanged -- every chunk goes into **both** indexes, because hybrid search needs both:

In [5]:
def ingest(documents, embed_fn, vector_index, keyword_index):
    for doc in documents:
        for chunk in chunk_doc(doc):
            vec = embed_fn(chunk["text"])
            vector_index.upsert(chunk["chunk_id"], vec, metadata=chunk)
            keyword_index.add(chunk["chunk_id"], chunk["text"], metadata=chunk)

This is the book's `search`, unchanged -- filter first, run both searches, merge by taking the max score per chunk, then rerank and threshold:

In [6]:
def search(query, user, embed_fn, vector_index, keyword_index, reranker):
    filters = {"tenant_id": user["tenant_id"], "access": {"$in": user["labels"]}}
    dense = vector_index.search(embed_fn(query), top_k=20, filters=filters)
    sparse = keyword_index.search(query, top_k=20, filters=filters)
    merged = {}
    for r in dense + sparse:
        merged[r["chunk_id"]] = max(merged.get(r["chunk_id"], 0), r["score"])
    candidates = [{"chunk_id": k, "score": v} for k, v in merged.items()]
    ranked = reranker.rerank(query, candidates)
    return [r for r in ranked if r["score"] >= 0.2][:5]

In [7]:
documents = [
    {
        "doc_id": "policy-1",
        "text": "Refunds are allowed within 30 days of purchase for annual plans. "
                "The refund window for enterprise plans is 45 days. ",
        "tenant_id": "acme", "source": "kb", "access": "support",
    },
]

vector_index, keyword_index = VectorIndex(), KeywordIndex()
ingest(documents, embed_fn, vector_index, keyword_index)
print(f"ingested {len(vector_index._store)} chunk(s) into both indexes")

chunk_lookup = {cid: entry["text"] for cid, entry in keyword_index._store.items()}
reranker = Reranker(chunk_lookup)
acme_support_user = {"tenant_id": "acme", "labels": ["support"]}

results = search("What is the refund window?", acme_support_user, embed_fn, vector_index, keyword_index, reranker)
print("standalone hybrid search:", results)

ingested 1 chunk(s) into both indexes
standalone hybrid search: [{'chunk_id': 'policy-1:0', 'score': 0.6}]


## Connecting through MCP
Inside LangGraph, this same retrieval becomes one node that calls an MCP tool instead of the local indexes directly -- the client no longer needs to know how search is implemented behind `search_docs`. A small server exposes it:

In [8]:
%%writefile production_rag_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("ProductionRAGServer")

_DOCS = [
    {"doc_id": "policy-1", "text": "Refunds are allowed within 30 days of purchase for annual plans. Enterprise plans follow a separate 45 day window.", "tenant_id": "acme", "source": "kb", "access": "support"},
    {"doc_id": "policy-2", "text": "Billing questions are answered by the support team within one business day. Escalate urgent billing issues to a manager.", "tenant_id": "acme", "source": "kb", "access": "support"},
]


@mcp.tool()
def search_docs(query: str, tenant: str) -> dict:
    """Search approved documents for a tenant."""
    q_words = query.lower().split()
    scored = []
    for doc in _DOCS:
        if doc["tenant_id"] != tenant:
            continue
        text_words = doc["text"].lower().split()
        overlap = sum(1 for qw in q_words if any(tw.startswith(qw) or qw.startswith(tw) for tw in text_words))
        if overlap > 0:
            scored.append({"doc_id": doc["doc_id"], "score": overlap, "text": doc["text"]})
    scored.sort(key=lambda r: r["score"], reverse=True)
    return {"results": scored[:5]}


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing production_rag_server.py


This is the book's `retrieve_node`, adapted in one necessary way: the book shows it as a plain `def`, but a real MCP tool call is awaitable, so it has to be `async def` to actually reach the server:

In [9]:
import sys, os

# Jupyter's stdout/stderr streams don't expose a real OS file descriptor, which crashes
# the MCP SDK's Windows subprocess-spawning fallback (it isn't needed outside Jupyter).
if os.name == "nt":
    try:
        sys.stderr.fileno()
    except Exception:
        sys.stderr = os.fdopen(os.dup(2), "w")

import asyncio
import json
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def call_tool(session: ClientSession, name: str, arguments: dict) -> Any:
    """Call an MCP tool and unwrap its result into a plain Python value."""
    result = await session.call_tool(name, arguments)
    if result.structuredContent is not None:
        content = result.structuredContent
        return content.get("result", content) if isinstance(content, dict) else content
    text = result.content[0].text
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

In [10]:
async def retrieve_node(state: dict, session: ClientSession) -> dict:
    hits = await call_tool(session, "search_docs", {"query": state["question"], "tenant": state["tenant_id"]})
    state["results"] = hits["results"]
    return state

In [11]:
server_path = Path.cwd() / "production_rag_server.py"
params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        state = {"question": "What is the refund window for annual plans?", "tenant_id": "acme"}
        state = await retrieve_node(state, session)

print("retrieve_node results (via MCP):")
for r in state["results"]:
    print(f"  {r['doc_id']}  score={r['score']}  {r['text'][:70]}...")

retrieve_node results (via MCP):
  policy-1  score=5  Refunds are allowed within 30 days of purchase for annual plans. Enter...
  policy-2  score=3  Billing questions are answered by the support team within one business...


That's the full path: chunk -> dual index -> filtered hybrid search -> rerank on the client side, or the same search reached through a standard MCP tool call. From here, an answer node (notebook 08's `generate_answer`) packs the top result with source labels and produces the final response.